# Copilot ML Training Report

Notebook reproducible for COP-014:
- load and prepare training data,
- train and evaluate with temporal split,
- generate ROC/PR/calibration and feature importance,
- export model + metadata + report artifacts.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, roc_curve


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ml' / 'train_copilot_model.py').exists():
            return candidate
    raise RuntimeError('Could not find project root containing ml/train_copilot_model.py')


ROOT = _find_project_root(Path.cwd().resolve())
if str(ROOT / 'ml') not in sys.path:
    sys.path.insert(0, str(ROOT / 'ml'))

from train_copilot_model import (  # noqa: E402
    FEATURES,
    build_training_frame,
    compute_train_window,
    train_with_evaluation,
)

print(f'Project root: {ROOT}')


In [ ]:
# Reproducible configuration.
EVENTS_ROOT = ROOT / 'data' / 'parquet_events'
MODEL_OUT = ROOT / 'data' / 'models' / 'copilot_model.joblib'
META_OUT = MODEL_OUT.with_suffix('.json')
REPORT_DIR = ROOT / 'data' / 'reports' / 'ml'

# By default, use full history (train_months=0).
TRAIN_START = os.getenv('TLC_MONTH') or None
TRAIN_MONTHS = int(os.getenv('TLC_TRAIN_MONTH_COUNT', '0') or '0')
RANDOM_STATE = 42
TEST_SIZE = 0.2
MIN_ROWS = 100

REVENUE_THRESHOLD_EUR_H = 18.0
CONTEXT_WINDOW_MINUTES = 20
POSITIVE_QUANTILE = 0.60

TRAIN_WINDOW = compute_train_window(TRAIN_START, TRAIN_MONTHS)
print('Events root:', EVENTS_ROOT)
print('Train window:', TRAIN_WINDOW[0].isoformat() if TRAIN_WINDOW else 'full-history', '->', TRAIN_WINDOW[1].isoformat() if TRAIN_WINDOW else 'full-history')
print('Model out:', MODEL_OUT)
print('Report dir:', REPORT_DIR)


In [ ]:
# 1) Load data and prepare training frame.
df = build_training_frame(
    events_root=EVENTS_ROOT,
    revenue_threshold_eur_h=REVENUE_THRESHOLD_EUR_H,
    context_window_minutes=CONTEXT_WINDOW_MINUTES,
    positive_quantile=POSITIVE_QUANTILE,
    train_window=TRAIN_WINDOW,
)

if df.empty:
    raise ValueError(f'No training rows found in {EVENTS_ROOT}')
if len(df) < MIN_ROWS:
    raise ValueError(f'Not enough rows to train: {len(df)} < {MIN_ROWS}')

frame = df.dropna(subset=FEATURES + ['label']).copy()

dataset_overview = pd.DataFrame(
    [
        {'metric': 'rows_total', 'value': len(df)},
        {'metric': 'rows_trainable', 'value': len(frame)},
        {'metric': 'positive_rate_global', 'value': float(frame['label'].mean())},
        {'metric': 'feature_count', 'value': len(FEATURES)},
        {'metric': 'time_min_utc', 'value': str(pd.to_datetime(frame['ts'], utc=True).min())},
        {'metric': 'time_max_utc', 'value': str(pd.to_datetime(frame['ts'], utc=True).max())},
        {'metric': 'label_threshold_eur_h_global', 'value': float(df['label_threshold_eur_h_global'].median())},
    ]
)

label_source_df = (
    frame['label_source']
    .value_counts()
    .rename_axis('label_source')
    .reset_index(name='count')
)

print('Label source distribution:')
print(label_source_df.to_string(index=False))

dataset_overview


In [ ]:
# 2) Train model using temporal split + lightweight regularization tuning.
model, metrics, evaluation = train_with_evaluation(df, random_state=RANDOM_STATE, test_size=TEST_SIZE)
y_test = evaluation['y_test']
proba = evaluation['proba_test']

temporal_metrics = {
    'roc_auc': float(metrics['roc_auc']),
    'average_precision': float(metrics['average_precision']),
    'brier_score': float(metrics['brier_score']),
    'ece_10_bins': float(metrics['ece_10_bins']),
    'test_rows': int(metrics['test_rows']),
    'positive_rate_test': float(metrics['positive_rate_test']),
}

random_split_metrics = {
    'roc_auc': float(metrics['random_split_roc_auc']),
    'average_precision': float(metrics['random_split_average_precision']),
}

quality_flags = list(metrics.get('quality_flags', []))
print('Quality flags:', quality_flags if quality_flags else ['none'])

metrics_df = pd.DataFrame(
    [
        {'metric': 'temporal_roc_auc', 'value': temporal_metrics['roc_auc']},
        {'metric': 'temporal_average_precision', 'value': temporal_metrics['average_precision']},
        {'metric': 'temporal_brier_score', 'value': temporal_metrics['brier_score']},
        {'metric': 'temporal_ece_10_bins', 'value': temporal_metrics['ece_10_bins']},
        {'metric': 'random_split_roc_auc', 'value': random_split_metrics['roc_auc']},
        {'metric': 'random_split_average_precision', 'value': random_split_metrics['average_precision']},
        {'metric': 'split_strategy', 'value': metrics['split_strategy']},
        {'metric': 'best_logreg_c', 'value': metrics['best_logreg_c']},
        {'metric': 'train_rows', 'value': metrics['train_rows']},
        {'metric': 'test_rows', 'value': metrics['test_rows']},
    ]
)
metrics_df


In [ ]:
# 3) ROC / PR / Calibration curves + PNG export.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

fpr, tpr, _ = roc_curve(y_test, proba)
precision_curve, recall_curve, _ = precision_recall_curve(y_test, proba)
cal_y, cal_p = calibration_curve(y_test, proba, n_bins=10, strategy='quantile')

fig_roc, ax_roc = plt.subplots(figsize=(5.2, 4.2))
ax_roc.plot(fpr, tpr, color='#0b7285', linewidth=2.0, label=f"AUC={temporal_metrics['roc_auc']:.3f}")
ax_roc.plot([0, 1], [0, 1], linestyle='--', color='#94a3b8', linewidth=1.2)
ax_roc.set_title('ROC Curve (Temporal Holdout)')
ax_roc.set_xlabel('False Positive Rate')
ax_roc.set_ylabel('True Positive Rate')
ax_roc.legend(loc='lower right')
ax_roc.grid(alpha=0.25)
roc_path = REPORT_DIR / 'copilot_roc_curve.png'
fig_roc.tight_layout()
fig_roc.savefig(roc_path, dpi=150, bbox_inches='tight')

fig_pr, ax_pr = plt.subplots(figsize=(5.2, 4.2))
ax_pr.plot(recall_curve, precision_curve, color='#15803d', linewidth=2.0, label=f"AP={temporal_metrics['average_precision']:.3f}")
ax_pr.set_title('Precision-Recall Curve (Temporal Holdout)')
ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precision')
ax_pr.legend(loc='lower left')
ax_pr.grid(alpha=0.25)
pr_path = REPORT_DIR / 'copilot_pr_curve.png'
fig_pr.tight_layout()
fig_pr.savefig(pr_path, dpi=150, bbox_inches='tight')

fig_cal, ax_cal = plt.subplots(figsize=(5.2, 4.2))
ax_cal.plot(cal_p, cal_y, marker='o', color='#b45309', linewidth=2.0, label=f"ECE={temporal_metrics['ece_10_bins']:.3f}")
ax_cal.plot([0, 1], [0, 1], linestyle='--', color='#94a3b8', linewidth=1.2)
ax_cal.set_title('Calibration Curve (Temporal Holdout)')
ax_cal.set_xlabel('Mean predicted probability')
ax_cal.set_ylabel('Observed frequency')
ax_cal.legend(loc='upper left')
ax_cal.grid(alpha=0.25)
cal_path = REPORT_DIR / 'copilot_calibration_curve.png'
fig_cal.tight_layout()
fig_cal.savefig(cal_path, dpi=150, bbox_inches='tight')

fig_combo, axes = plt.subplots(1, 3, figsize=(15.5, 4.2))
axes[0].plot(fpr, tpr, color='#0b7285', linewidth=1.9)
axes[0].plot([0, 1], [0, 1], linestyle='--', color='#94a3b8', linewidth=1.0)
axes[0].set_title('ROC')
axes[1].plot(recall_curve, precision_curve, color='#15803d', linewidth=1.9)
axes[1].set_title('PR')
axes[2].plot(cal_p, cal_y, marker='o', color='#b45309', linewidth=1.9)
axes[2].plot([0, 1], [0, 1], linestyle='--', color='#94a3b8', linewidth=1.0)
axes[2].set_title('Calibration')
for ax in axes:
    ax.grid(alpha=0.2)
combo_path = REPORT_DIR / 'copilot_training_curves.png'
fig_combo.tight_layout()
fig_combo.savefig(combo_path, dpi=150, bbox_inches='tight')

plt.show()

print('Saved:', roc_path)
print('Saved:', pr_path)
print('Saved:', cal_path)
print('Saved:', combo_path)


In [ ]:
# 4) Feature importance from logistic coefficients.
coef = pd.Series(model.named_steps['clf'].coef_[0], index=FEATURES)
importance_df = pd.DataFrame({
    'feature': coef.index,
    'coefficient': coef.values,
    'abs_importance': coef.abs().values,
}).sort_values('abs_importance', ascending=False)

fig_imp, ax_imp = plt.subplots(figsize=(8.2, 5.2))
colors = ['#0b7285' if value >= 0 else '#b42318' for value in importance_df['coefficient']]
ax_imp.barh(importance_df['feature'], importance_df['coefficient'], color=colors, alpha=0.88)
ax_imp.invert_yaxis()
ax_imp.set_title('Feature Importance (Logistic Coefficients)')
ax_imp.set_xlabel('Coefficient value')
ax_imp.grid(axis='x', alpha=0.2)
importance_path = REPORT_DIR / 'copilot_feature_importance.png'
fig_imp.tight_layout()
fig_imp.savefig(importance_path, dpi=150, bbox_inches='tight')
plt.show()

print('Saved:', importance_path)
importance_df


In [ ]:
# 5) Export model + metadata to data/models.
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
trained_at_utc = datetime.now(timezone.utc).isoformat()
label_source_counts = {str(k): int(v) for k, v in frame['label_source'].value_counts().to_dict().items()}
train_window_signature = 'full' if TRAIN_WINDOW is None else f"{TRAIN_WINDOW[0].isoformat()}::{TRAIN_WINDOW[1].isoformat()}"

model_payload = {
    'model': model,
    'feature_columns': FEATURES,
    'trained_rows': int(len(df)),
    'positive_rate': float(frame['label'].mean()),
    'label_threshold_eur_h': float(df['label_threshold_eur_h_global'].median()),
    'base_threshold_eur_h': float(REVENUE_THRESHOLD_EUR_H),
    'positive_quantile': float(POSITIVE_QUANTILE),
    'context_window_minutes': int(CONTEXT_WINDOW_MINUTES),
    'label_source_counts': label_source_counts,
    'metrics': metrics,
    'quality_flags': quality_flags,
    'model_version': 'copilot_v2',
    'trained_at_utc': trained_at_utc,
    'train_window_start': TRAIN_WINDOW[0].isoformat() if TRAIN_WINDOW else None,
    'train_window_end': TRAIN_WINDOW[1].isoformat() if TRAIN_WINDOW else None,
    'train_window_signature': train_window_signature,
}
joblib.dump(model_payload, MODEL_OUT)

meta_payload = {
    'feature_columns': FEATURES,
    'trained_rows': int(len(df)),
    'positive_rate': round(float(frame['label'].mean()), 4),
    'label_threshold_eur_h': float(df['label_threshold_eur_h_global'].median()),
    'base_threshold_eur_h': float(REVENUE_THRESHOLD_EUR_H),
    'positive_quantile': float(POSITIVE_QUANTILE),
    'context_window_minutes': int(CONTEXT_WINDOW_MINUTES),
    'label_source_counts': label_source_counts,
    'metrics': metrics,
    'quality_flags': quality_flags,
    'model_version': 'copilot_v2',
    'trained_at_utc': trained_at_utc,
    'train_window_start': TRAIN_WINDOW[0].isoformat() if TRAIN_WINDOW else None,
    'train_window_end': TRAIN_WINDOW[1].isoformat() if TRAIN_WINDOW else None,
    'train_window_signature': train_window_signature,
}
META_OUT.write_text(json.dumps(meta_payload, indent=2), encoding='utf-8')

print('Model exported:', MODEL_OUT)
print('Metadata exported:', META_OUT)


In [ ]:
# 6) Export mini report to data/reports/ml.
report_json_path = REPORT_DIR / 'copilot_training_report.json'
report_md_path = REPORT_DIR / 'copilot_training_report.md'

report_payload = {
    'generated_at_utc': trained_at_utc,
    'inputs': {
        'events_root': str(EVENTS_ROOT),
        'train_start': TRAIN_START,
        'train_months': TRAIN_MONTHS,
        'revenue_threshold_eur_h': REVENUE_THRESHOLD_EUR_H,
        'context_window_minutes': CONTEXT_WINDOW_MINUTES,
        'positive_quantile': POSITIVE_QUANTILE,
        'test_size': TEST_SIZE,
        'random_state': RANDOM_STATE,
    },
    'dataset': {
        'rows_total': int(len(df)),
        'rows_trainable': int(len(frame)),
        'rows_train_split': int(metrics['train_rows']),
        'rows_test_split': int(metrics['test_rows']),
        'positive_rate_global': float(frame['label'].mean()),
        'label_threshold_eur_h': float(df['label_threshold_eur_h_global'].median()),
        'label_source_counts': label_source_counts,
    },
    'metrics': metrics,
    'quality_flags': quality_flags,
    'top_features': importance_df.head(10).to_dict(orient='records'),
    'artifacts': {
        'model_joblib': str(MODEL_OUT),
        'model_metadata_json': str(META_OUT),
        'roc_curve_png': str(roc_path),
        'pr_curve_png': str(pr_path),
        'calibration_curve_png': str(cal_path),
        'combined_curves_png': str(combo_path),
        'feature_importance_png': str(importance_path),
    },
}
report_json_path.write_text(json.dumps(report_payload, indent=2), encoding='utf-8')

md_lines = [
    '# Copilot Training Report',
    '',
    f'- Generated at (UTC): {trained_at_utc}',
    f'- Events root: {EVENTS_ROOT}',
    f"- Train window: {'full-history' if TRAIN_WINDOW is None else TRAIN_WINDOW[0].isoformat() + ' -> ' + TRAIN_WINDOW[1].isoformat()}",
    '',
    '## Dataset',
    f'- Rows total: {len(df)}',
    f"- Rows train split: {metrics['train_rows']}",
    f"- Rows test split: {metrics['test_rows']}",
    f"- Positive rate: {float(frame['label'].mean()):.4f}",
    f"- Label threshold EUR/h (global): {float(df['label_threshold_eur_h_global'].median()):.2f}",
    '',
    '## Temporal Metrics',
    f"- ROC AUC: {metrics['roc_auc']:.4f}",
    f"- Average Precision: {metrics['average_precision']:.4f}",
    f"- Brier score: {metrics['brier_score']:.4f}",
    f"- ECE (10 bins): {metrics['ece_10_bins']:.4f}",
    f"- Precision@0.5: {metrics['precision_at_0_5']:.4f}",
    f"- Recall@0.5: {metrics['recall_at_0_5']:.4f}",
    f"- F1@0.5: {metrics['f1_at_0_5']:.4f}",
    '',
    '## Diagnostics',
    f"- Split strategy: {metrics['split_strategy']}",
    f"- Best logistic C: {metrics['best_logreg_c']}",
    f"- Random split ROC AUC: {metrics['random_split_roc_auc']:.4f}",
    f"- Random split AP: {metrics['random_split_average_precision']:.4f}",
    f"- Quality flags: {', '.join(quality_flags) if quality_flags else 'none'}",
    '',
    '## Top Features (abs coefficient)',
]
for row in importance_df.head(10).itertuples(index=False):
    md_lines.append(f'- {row.feature}: coef={row.coefficient:.4f} abs={row.abs_importance:.4f}')

md_lines.extend([
    '',
    '## Artifacts',
    f'- Model: {MODEL_OUT}',
    f'- Metadata: {META_OUT}',
    f'- ROC curve: {roc_path}',
    f'- PR curve: {pr_path}',
    f'- Calibration curve: {cal_path}',
    f'- Combined curves: {combo_path}',
    f'- Feature importance: {importance_path}',
    f'- Mini report JSON: {report_json_path}',
])
report_md_path.write_text('\n'.join(md_lines), encoding='utf-8')

print('Mini report JSON:', report_json_path)
print('Mini report Markdown:', report_md_path)


In [ ]:
# 7) Final artifact summary for demo.
summary = pd.DataFrame(
    [
        {'artifact': 'model_joblib', 'path': str(MODEL_OUT)},
        {'artifact': 'model_metadata_json', 'path': str(META_OUT)},
        {'artifact': 'report_json', 'path': str(report_json_path)},
        {'artifact': 'report_markdown', 'path': str(report_md_path)},
        {'artifact': 'roc_curve_png', 'path': str(roc_path)},
        {'artifact': 'pr_curve_png', 'path': str(pr_path)},
        {'artifact': 'calibration_curve_png', 'path': str(cal_path)},
        {'artifact': 'feature_importance_png', 'path': str(importance_path)},
    ]
)
summary
